In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import cv2
import numpy as np

class MNISTEdgeDataset(Dataset):
    """MNIST with edge map conditioning"""
    
    def __init__(self, train=True, conditioning='edge'):
        self.mnist = torchvision.datasets.MNIST(
            root='./data', 
            train=train, 
            download=True
        )
        self.conditioning = conditioning
        
    def __len__(self):
        return len(self.mnist)
    
    def get_edge_map(self, img):
        """Generate Canny edge map"""
        img_np = np.array(img)
        edges = cv2.Canny(img_np, 50, 150)
        return edges
    
    def get_skeleton(self, img):
        """Generate morphological skeleton"""
        img_np = np.array(img)
        _, binary = cv2.threshold(img_np, 127, 255, cv2.THRESH_BINARY)
        skeleton = cv2.ximgproc.thinning(binary)
        return skeleton
    
    def __getitem__(self, idx):
        img, label = self.mnist[idx]
        
        # Convert to tensor [1, 28, 28], normalize to [-1, 1]
        img_tensor = transforms.ToTensor()(img)
        img_tensor = (img_tensor - 0.5) / 0.5
        
        # Generate conditioning signal
        if self.conditioning == 'edge':
            condition = self.get_edge_map(img)
        elif self.conditioning == 'skeleton':
            condition = self.get_skeleton(img)
        else:
            raise ValueError(f"Unknown conditioning: {self.conditioning}")
        
        # Normalize condition to [-1, 1]
        condition_tensor = torch.from_numpy(condition).float().unsqueeze(0) / 127.5 - 1.0
        
        return {
            'image': img_tensor,
            'condition': condition_tensor,
            'label': label
        }

def get_dataloader(batch_size=64, train=True, conditioning='edge'):
    dataset = MNISTEdgeDataset(train=train, conditioning=conditioning)
    return torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=train,
        num_workers=4
    )

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================================
# LoRA Layer
# ============================================================================

class LoRALayer(nn.Module):
    """Low-Rank Adaptation layer"""
    def __init__(self, in_features, out_features, rank=4):
        super().__init__()
        self.rank = rank
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.scaling = 1.0
        
    def forward(self, x):
        # x: [B, in_features] or [B, C, H, W]
        if x.dim() == 4:
            B, C, H, W = x.shape
            x_flat = x.permute(0, 2, 3, 1).reshape(-1, C)  # [B*H*W, C]
            delta = (x_flat @ self.lora_A @ self.lora_B) * self.scaling
            delta = delta.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        else:
            delta = (x @ self.lora_A @ self.lora_B) * self.scaling
        return delta

# ============================================================================
# Basic UNet Blocks
# ============================================================================

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        
        if in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x, t_emb):
        h = self.conv1(F.relu(x))
        h = self.norm1(h)
        
        # Add time embedding
        h = h + self.time_mlp(F.relu(t_emb))[:, :, None, None]
        
        h = self.conv2(F.relu(h))
        h = self.norm2(h)
        
        return h + self.shortcut(x)

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)
        
        # Reshape for attention
        q = q.reshape(B, C, H * W).permute(0, 2, 1)  # [B, HW, C]
        k = k.reshape(B, C, H * W)  # [B, C, HW]
        v = v.reshape(B, C, H * W).permute(0, 2, 1)  # [B, HW, C]
        
        attn = torch.softmax(q @ k / (C ** 0.5), dim=-1)  # [B, HW, HW]
        out = attn @ v  # [B, HW, C]
        out = out.permute(0, 2, 1).reshape(B, C, H, W)
        
        return x + self.proj(out)

# ============================================================================
# Simple UNet (Base Denoiser)
# ============================================================================

class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, model_channels=64, time_dim=256):
        super().__init__()
        
        self.time_dim = time_dim
        self.time_embed = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU()
        )
        
        # Encoder
        self.conv_in = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        
        self.down1 = nn.ModuleList([
            ResBlock(model_channels, model_channels, time_dim),
            ResBlock(model_channels, model_channels, time_dim)
        ])
        self.downsample1 = nn.Conv2d(model_channels, model_channels, 3, stride=2, padding=1)
        
        self.down2 = nn.ModuleList([
            ResBlock(model_channels, model_channels * 2, time_dim),
            ResBlock(model_channels * 2, model_channels * 2, time_dim)
        ])
        self.downsample2 = nn.Conv2d(model_channels * 2, model_channels * 2, 3, stride=2, padding=1)
        
        # Middle
        self.mid = nn.ModuleList([
            ResBlock(model_channels * 2, model_channels * 2, time_dim),
            AttentionBlock(model_channels * 2),
            ResBlock(model_channels * 2, model_channels * 2, time_dim)
        ])
        
        # Decoder
        self.upsample2 = nn.ConvTranspose2d(model_channels * 2, model_channels * 2, 4, stride=2, padding=1)
        self.up2 = nn.ModuleList([
            ResBlock(model_channels * 4, model_channels * 2, time_dim),
            ResBlock(model_channels * 2, model_channels, time_dim)
        ])
        
        self.upsample1 = nn.ConvTranspose2d(model_channels, model_channels, 4, stride=2, padding=1)
        self.up1 = nn.ModuleList([
            ResBlock(model_channels * 2, model_channels, time_dim),
            ResBlock(model_channels, model_channels, time_dim)
        ])
        
        self.conv_out = nn.Conv2d(model_channels, in_channels, 3, padding=1)
    
    def forward(self, x, t):
        # Time embedding
        t_emb = self.time_embed(t)
        
        # Initial conv
        h = self.conv_in(x)
        
        # Encoder
        h1 = h
        for block in self.down1:
            h1 = block(h1, t_emb)
        h1_skip = h1
        h1 = self.downsample1(h1)
        
        h2 = h1
        for block in self.down2:
            h2 = block(h2, t_emb)
        h2_skip = h2
        h2 = self.downsample2(h2)
        
        # Middle
        h = h2
        for block in self.mid:
            if isinstance(block, AttentionBlock):
                h = block(h)
            else:
                h = block(h, t_emb)
        
        # Decoder
        h = self.upsample2(h)
        h = torch.cat([h, h2_skip], dim=1)
        for block in self.up2:
            h = block(h, t_emb)
        
        h = self.upsample1(h)
        h = torch.cat([h, h1_skip], dim=1)
        for block in self.up1:
            h = block(h, t_emb)
        
        return self.conv_out(h)

# ============================================================================
# ControlNet (Full Copy)
# ============================================================================

class ControlNet(nn.Module):
    """Full ControlNet: copy of UNet encoder + zero convs"""
    def __init__(self, base_unet, conditioning_channels=1):
        super().__init__()
        
        # Copy encoder from base UNet
        self.time_embed = base_unet.time_embed
        self.conv_in = nn.Conv2d(conditioning_channels, 64, 3, padding=1)
        
        # Copy encoder blocks
        self.down1 = nn.ModuleList([
            ResBlock(64, 64, 256),
            ResBlock(64, 64, 256)
        ])
        self.downsample1 = nn.Conv2d(64, 64, 3, stride=2, padding=1)
        
        self.down2 = nn.ModuleList([
            ResBlock(64, 128, 256),
            ResBlock(128, 128, 256)
        ])
        self.downsample2 = nn.Conv2d(128, 128, 3, stride=2, padding=1)
        
        self.mid = nn.ModuleList([
            ResBlock(128, 128, 256),
            AttentionBlock(128),
            ResBlock(128, 128, 256)
        ])
        
        # Zero convolutions for control injection
        self.zero_convs = nn.ModuleList([
            nn.Conv2d(64, 64, 1),
            nn.Conv2d(64, 64, 1),
            nn.Conv2d(128, 128, 1),
            nn.Conv2d(128, 128, 1),
            nn.Conv2d(128, 128, 1),
        ])
        
        # Initialize zero convs to zero
        for zc in self.zero_convs:
            nn.init.zeros_(zc.weight)
            nn.init.zeros_(zc.bias)
    
    def forward(self, condition, t):
        t_emb = self.time_embed(t)
        
        controls = []
        h = self.conv_in(condition)
        
        # Down1
        for block in self.down1:
            h = block(h, t_emb)
        controls.append(self.zero_convs[0](h))
        h = self.downsample1(h)
        controls.append(self.zero_convs[1](h))
        
        # Down2
        for block in self.down2:
            h = block(h, t_emb)
        controls.append(self.zero_convs[2](h))
        h = self.downsample2(h)
        controls.append(self.zero_convs[3](h))
        
        # Mid
        for block in self.mid:
            if isinstance(block, AttentionBlock):
                h = block(h)
            else:
                h = block(h, t_emb)
        controls.append(self.zero_convs[4](h))
        
        return controls

# ============================================================================
# LoRA-ControlNet (Your Hypothesis)
# ============================================================================

class LoRAControlNet(nn.Module):
    """ControlNet using LoRA instead of full copy"""
    def __init__(self, base_unet, conditioning_channels=1, rank=4, inject_layer='mid'):
        super().__init__()
        
        self.inject_layer = inject_layer
        self.rank = rank
        
        # Lightweight condition encoder
        self.condition_encoder = nn.Sequential(
            nn.Conv2d(conditioning_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU()
        )
        
        # Always create projections (cleaner approach)
        self.proj_28 = nn.Conv2d(64, 64, 1)   # For 28x28 scale
        self.proj_14 = nn.Conv2d(64, 128, 1)  # For 14x14 scale
        self.proj_7 = nn.Conv2d(64, 128, 1)   # For 7x7 scale
        
        # LoRA injections based on strategy
        if inject_layer == 'mid':
            self.lora_mid = LoRALayer(128, 128, rank)
            
        elif inject_layer == 'attention':
            self.lora_attn = LoRALayer(128, 128, rank)
            
        elif inject_layer == 'all':
            self.lora_down1 = LoRALayer(64, 64, rank)
            self.lora_down2 = LoRALayer(128, 128, rank)
            self.lora_mid = LoRALayer(128, 128, rank)
    
    def forward(self, condition, t):
        # Encode condition
        cond_feat = self.condition_encoder(condition)  # [B, 64, 28, 28]
        
        # Multi-scale features
        cond_28 = self.proj_28(cond_feat)  # [B, 64, 28, 28]
        cond_14 = self.proj_14(F.interpolate(cond_feat, scale_factor=0.5))  # [B, 128, 14, 14]
        cond_7 = self.proj_7(F.interpolate(cond_feat, scale_factor=0.25))  # [B, 128, 7, 7]
        
        if self.inject_layer == 'mid':
            delta = self.lora_mid(cond_7)
            controls = [None, None, None, None, delta]
            
        elif self.inject_layer == 'attention':
            delta = self.lora_attn(cond_7)
            controls = [None, None, None, None, delta]
            
        elif self.inject_layer == 'all':
            delta1 = self.lora_down1(cond_28)
            delta2 = self.lora_down2(cond_14)
            delta_mid = self.lora_mid(cond_7)
            controls = [delta1, None, delta2, None, delta_mid]
        
        return controls
# ============================================================================
# Conditional UNet (combines base UNet + ControlNet/LoRA)
# ============================================================================

class ConditionalUNet(nn.Module):
    def __init__(self, base_unet, control_net):
        super().__init__()
        self.base_unet = base_unet
        self.control_net = control_net
    
    def forward(self, x, t, condition):
        # Get control signals
        controls = self.control_net(condition, t)
        
        # Modified UNet forward with control injection
        t_emb = self.base_unet.time_embed(t)
        h = self.base_unet.conv_in(x)
        
        # Encoder with control injection
        h1 = h
        for block in self.base_unet.down1:
            h1 = block(h1, t_emb)
        if controls[0] is not None:
            h1 = h1 + controls[0]
        h1_skip = h1
        h1 = self.base_unet.downsample1(h1)
        if controls[1] is not None:
            h1 = h1 + controls[1]
        
        h2 = h1
        for block in self.base_unet.down2:
            h2 = block(h2, t_emb)
        if controls[2] is not None:
            h2 = h2 + controls[2]
        h2_skip = h2
        h2 = self.base_unet.downsample2(h2)
        if controls[3] is not None:
            h2 = h2 + controls[3]
        
        # Middle with control
        h = h2
        for block in self.base_unet.mid:
            if isinstance(block, AttentionBlock):
                h = block(h)
            else:
                h = block(h, t_emb)
        if controls[4] is not None:
            h = h + controls[4]
        
        # Decoder (no control injection)
        h = self.base_unet.upsample2(h)
        h = torch.cat([h, h2_skip], dim=1)
        for block in self.base_unet.up2:
            h = block(h, t_emb)
        
        h = self.base_unet.upsample1(h)
        h = torch.cat([h, h1_skip], dim=1)
        for block in self.base_unet.up1:
            h = block(h, t_emb)
        
        return self.base_unet.conv_out(h)

# ============================================================================
# Model Factory
# ============================================================================

def create_model(model_type='controlnet', rank=4, inject_layer='mid'):
    """
    model_type: 'baseline', 'controlnet', 'lora_mid', 'lora_attention', 'lora_all'
    """
    base_unet = SimpleUNet()
    
    if model_type == 'baseline':
        return base_unet
    
    elif model_type == 'controlnet':
        control_net = ControlNet(base_unet)
        return ConditionalUNet(base_unet, control_net)
    
    elif model_type.startswith('lora'):
        layer = inject_layer if model_type == 'lora' else model_type.split('_')[1]
        control_net = LoRAControlNet(base_unet, rank=rank, inject_layer=layer)
        return ConditionalUNet(base_unet, control_net)
    
    else:
        raise ValueError(f"Unknown model type: {model_type}")

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from model import create_model
from dataset import get_dataloader
import os

# ============================================================================
# Diffusion Utilities
# ============================================================================

def linear_beta_schedule(timesteps=1000, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def get_index_from_list(vals, t, x_shape):
    """Fixed version - no .cpu() call"""
    batch_size = t.shape[0]
    out = vals.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

class DiffusionTrainer:
    def __init__(self, model, device='cuda', timesteps=1000):
        self.model = model.to(device)
        self.device = device
        self.timesteps = timesteps
        
        # Define beta schedule - ensure they're on the correct device
        self.betas = linear_beta_schedule(timesteps).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
    
    def q_sample(self, x_start, t, noise=None):
        """Forward diffusion: add noise to x_start"""
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alpha_cumprod_t = get_index_from_list(self.sqrt_alphas_cumprod, t, x_start.shape)
        sqrt_one_minus_alpha_cumprod_t = get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x_start.shape
        )
        
        return sqrt_alpha_cumprod_t * x_start + sqrt_one_minus_alpha_cumprod_t * noise
    
    def train_step(self, batch, optimizer):
        optimizer.zero_grad()
        
        x = batch['image'].to(self.device)
        condition = batch['condition'].to(self.device)
        
        # Sample random timesteps
        t = torch.randint(0, self.timesteps, (x.shape[0],), device=self.device).long()
        
        # Add noise
        noise = torch.randn_like(x)
        x_noisy = self.q_sample(x, t, noise)
        
        # Predict noise
        if isinstance(self.model, nn.Module) and hasattr(self.model, 'control_net'):
            noise_pred = self.model(x_noisy, t, condition)
        else:
            noise_pred = self.model(x_noisy, t)
        
        # Loss
        loss = nn.functional.mse_loss(noise_pred, noise)
        
        loss.backward()
        optimizer.step()
        
        return loss.item()
    
    @torch.no_grad()
    def sample(self, condition, n_samples=8):
        """DDPM sampling"""
        self.model.eval()
        
        # Start from noise
        x = torch.randn(n_samples, 1, 28, 28).to(self.device)
        condition = condition.to(self.device)
        
        for i in reversed(range(self.timesteps)):
            t = torch.full((n_samples,), i, device=self.device, dtype=torch.long)
            
            # Predict noise
            if hasattr(self.model, 'control_net'):
                noise_pred = self.model(x, t, condition)
            else:
                noise_pred = self.model(x, t)
            
            # Get coefficients
            beta_t = get_index_from_list(self.betas, t, x.shape)
            alpha_t = get_index_from_list(self.alphas, t, x.shape)
            alpha_cumprod_t = get_index_from_list(self.alphas_cumprod, t, x.shape)
            
            # DDPM update
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
            
            x = (1 / torch.sqrt(alpha_t)) * (
                x - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * noise_pred
            ) + torch.sqrt(beta_t) * noise
        
        self.model.train()
        return x

# ============================================================================
# Training Function
# ============================================================================

def train(model_type='lora', rank=4, inject_layer='attention', epochs=5, batch_size=128, lr=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on {device}")
    
    # Create model
    model = create_model(model_type, rank=rank, inject_layer=inject_layer)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model_type} ({inject_layer})")
    print(f"Rank: {rank}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    
    # Data
    train_loader = get_dataloader(batch_size=batch_size, train=True)
    
    # Training
    trainer = DiffusionTrainer(model, device=device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in pbar:
            loss = trainer.train_step(batch, optimizer)
            epoch_loss += loss
            pbar.set_postfix({'loss': f'{loss:.4f}'})
        
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")
        
        # Save checkpoint
        os.makedirs('checkpoints', exist_ok=True)
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'loss': avg_loss
        }, f'checkpoints/{model_type}_rank{rank}_{inject_layer}_epoch{epoch}.pt')
    
    return model, trainer

# ============================================================================
# Main
# ============================================================================

if __name__ == '__main__':
    print("\n" + "="*50)
    print("Experiment 3: LoRA (Attention, rank=4)")
    print("="*50)
    
    model_lora_attn, trainer_lora_attn = train(
        model_type='lora', 
        rank=4, 
        inject_layer='attention', 
        epochs=5
    )
    
    print("\n✅ Training complete!")
    print("Checkpoint saved to: checkpoints/lora_rank4_attention_epoch4.pt")


Experiment 3: LoRA (Attention, rank=4)
Training on cuda
Model: lora (attention)
Rank: 4
Total params: 2,966,401
Trainable params: 2,966,401


Epoch 1/5:   0%|          | 0/469 [00:07<?, ?it/s]


AttributeError: 'LoRAControlNet' object has no attribute 'proj_mid'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
# from model import create_model
# from dataset import get_dataloader
import os

# ============================================================================
# Diffusion Utilities
# ============================================================================

def linear_beta_schedule(timesteps=1000, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def get_index_from_list(vals, t, x_shape):
    """Fixed version - no .cpu() call"""
    batch_size = t.shape[0]
    out = vals.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

class DiffusionTrainer:
    def __init__(self, model, device='cuda', timesteps=1000):
        self.model = model.to(device)
        self.device = device
        self.timesteps = timesteps
        
        # Define beta schedule - ensure they're on the correct device
        self.betas = linear_beta_schedule(timesteps).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
    
    def q_sample(self, x_start, t, noise=None):
        """Forward diffusion: add noise to x_start"""
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alpha_cumprod_t = get_index_from_list(self.sqrt_alphas_cumprod, t, x_start.shape)
        sqrt_one_minus_alpha_cumprod_t = get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x_start.shape
        )
        
        return sqrt_alpha_cumprod_t * x_start + sqrt_one_minus_alpha_cumprod_t * noise
    
    def train_step(self, batch, optimizer):
        optimizer.zero_grad()
        
        x = batch['image'].to(self.device)
        condition = batch['condition'].to(self.device)
        
        # Sample random timesteps
        t = torch.randint(0, self.timesteps, (x.shape[0],), device=self.device).long()
        
        # Add noise
        noise = torch.randn_like(x)
        x_noisy = self.q_sample(x, t, noise)
        
        # Predict noise
        if isinstance(self.model, nn.Module) and hasattr(self.model, 'control_net'):
            noise_pred = self.model(x_noisy, t, condition)
        else:
            noise_pred = self.model(x_noisy, t)
        
        # Loss
        loss = nn.functional.mse_loss(noise_pred, noise)
        
        loss.backward()
        optimizer.step()
        
        return loss.item()
    
    @torch.no_grad()
    def sample(self, condition, n_samples=8):
        """DDPM sampling"""
        self.model.eval()
        
        # Start from noise
        x = torch.randn(n_samples, 1, 28, 28).to(self.device)
        condition = condition.to(self.device)
        
        for i in reversed(range(self.timesteps)):
            t = torch.full((n_samples,), i, device=self.device, dtype=torch.long)
            
            # Predict noise
            if hasattr(self.model, 'control_net'):
                noise_pred = self.model(x, t, condition)
            else:
                noise_pred = self.model(x, t)
            
            # Get coefficients
            beta_t = get_index_from_list(self.betas, t, x.shape)
            alpha_t = get_index_from_list(self.alphas, t, x.shape)
            alpha_cumprod_t = get_index_from_list(self.alphas_cumprod, t, x.shape)
            
            # DDPM update
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
            
            x = (1 / torch.sqrt(alpha_t)) * (
                x - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * noise_pred
            ) + torch.sqrt(beta_t) * noise
        
        self.model.train()
        return x

# ============================================================================
# Training Loop
# ============================================================================

def train(model_type='controlnet', rank=4, inject_layer='mid', epochs=10, batch_size=128, lr=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on {device}")
    
    # Create model
    model = create_model(model_type, rank=rank, inject_layer=inject_layer)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model_type}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    
    # Data
    train_loader = get_dataloader(batch_size=batch_size, train=True)
    
    # Training
    trainer = DiffusionTrainer(model, device=device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in pbar:
            loss = trainer.train_step(batch, optimizer)
            epoch_loss += loss
            pbar.set_postfix({'loss': f'{loss:.4f}'})
        
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")
        
        # Save checkpoint
        os.makedirs('checkpoints', exist_ok=True)
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'loss': avg_loss
        }, f'checkpoints/{model_type}_rank{rank}_{inject_layer}_epoch{epoch}.pt')
    
    return model, trainer

if __name__ == '__main__':
    # Experiment 1: Full ControlNet
    print("\n" + "="*50)
    print("Experiment 1: Full ControlNet")
    print("="*50)
    model_cn, trainer_cn = train(model_type='controlnet', epochs=5)
    
    # Experiment 2: LoRA Mid-block
    print("\n" + "="*50)
    print("Experiment 2: LoRA (Mid-block, rank=4)")
    print("="*50)
    model_lora_mid, trainer_lora_mid = train(model_type='lora', rank=4, inject_layer='mid', epochs=5)
    
    # Experiment 3: LoRA Attention
    print("\n" + "="*50)
    print("Experiment 3: LoRA (Attention, rank=4)")
    print("="*50)
    model_lora_attn, trainer_lora_attn = train(model_type='lora', rank=4, inject_layer='attention', epochs=5)


Experiment 1: Full ControlNet
Training on cuda
Model: controlnet
Total params: 4,685,889
Trainable params: 4,685,889
Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:03<00:00, 3.05MB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 312kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



...

Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 1.01MB/s]


Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



Epoch 1/5:   0%|          | 0/469 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 87144, 35328, 73556, 160040) exited unexpectedly

In [10]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
import cv2
from model import create_model
from dataset import get_dataloader
from train import DiffusionTrainer

# ============================================================================
# Evaluation Metrics
# ============================================================================

def edge_alignment_score(generated, condition):
    """
    Measure how well generated images align with edge conditioning
    """
    gen_np = generated.cpu().numpy()
    cond_np = condition.cpu().numpy()
    
    scores = []
    for i in range(gen_np.shape[0]):
        gen_img = ((gen_np[i, 0] + 1) * 127.5).astype(np.uint8)
        cond_img = ((cond_np[i, 0] + 1) * 127.5).astype(np.uint8)
        
        # Extract edges from generated
        gen_edges = cv2.Canny(gen_img, 50, 150)
        
        # Compute overlap
        overlap = np.sum((gen_edges > 0) & (cond_img > 0))
        total_cond = np.sum(cond_img > 0)
        
        if total_cond > 0:
            scores.append(overlap / total_cond)
        else:
            scores.append(0.0)
    
    return np.mean(scores)

def compute_mse(generated, target):
    """MSE between generated and target images"""
    return mean_squared_error(
        target.cpu().numpy().flatten(),
        generated.cpu().numpy().flatten()
    )

# ============================================================================
# Visualization
# ============================================================================

def visualize_samples(model, trainer, test_loader, n_samples=8, save_path='samples.png'):
    """Generate and visualize samples"""
    model.eval()
    
    # Get test batch
    batch = next(iter(test_loader))
    condition = batch['condition'][:n_samples]
    target = batch['image'][:n_samples]
    
    # Sample
    with torch.no_grad():
        generated = trainer.sample(condition, n_samples=n_samples)
    
    # Plot
    fig, axes = plt.subplots(3, n_samples, figsize=(n_samples * 2, 6))
    
    for i in range(n_samples):
        # Condition
        axes[0, i].imshow(condition[i, 0].cpu().numpy(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Condition (Edges)', fontsize=10)
        
        # Generated
        axes[1, i].imshow(generated[i, 0].cpu().numpy(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Generated', fontsize=10)
        
        # Target
        axes[2, i].imshow(target[i, 0].cpu().numpy(), cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_title('Target', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved visualization to {save_path}")

# ============================================================================
# Full Evaluation
# ============================================================================

def evaluate_model(checkpoint_path, model_type, rank=4, inject_layer='mid'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model
    model = create_model(model_type, rank=rank, inject_layer=inject_layer)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    model = model.to(device)
    
    trainer = DiffusionTrainer(model, device=device)
    
    # Get test data
    test_loader = get_dataloader(batch_size=64, train=False)
    
    # Compute metrics
    print(f"\nEvaluating {model_type}...")
    
    all_edge_scores = []
    all_mse = []
    
    for batch in test_loader:
        condition = batch['condition'].to(device)
        target = batch['image'].to(device)
        
        with torch.no_grad():
            generated = trainer.sample(condition, n_samples=condition.shape[0])
        
        # Metrics
        edge_score = edge_alignment_score(generated, condition)
        mse = compute_mse(generated, target)
        
        all_edge_scores.append(edge_score)
        all_mse.append(mse)
        
        break  # For speed, evaluate on one batch
    
    print(f"Edge Alignment Score: {np.mean(all_edge_scores):.4f}")
    print(f"MSE: {np.mean(all_mse):.4f}")
    
    # Visualize
    visualize_samples(
        model, trainer, test_loader, 
        save_path=f'samples_{model_type}_rank{rank}_{inject_layer}.png'
    )
    
    return {
        'edge_score': np.mean(all_edge_scores),
        'mse': np.mean(all_mse)
    }

# ============================================================================
# Comparative Analysis
# ============================================================================

def compare_all_models():
    """Compare all trained models"""
    
    results = {}
    
    # ControlNet
    results['ControlNet'] = evaluate_model(
        'checkpoints/controlnet_rank4_mid_epoch4.pt',
        'controlnet'
    )
    
    # LoRA variants
    for layer in ['mid', 'attention', 'all']:
        results[f'LoRA_{layer}'] = evaluate_model(
            f'checkpoints/lora_rank4_{layer}_epoch4.pt',
            'lora',
            rank=4,
            inject_layer=layer
        )
    
    # Print comparison table
    print("\n" + "="*60)
    print("RESULTS COMPARISON")
    print("="*60)
    print(f"{'Model':<20} {'Edge Score':<15} {'MSE':<15}")
    print("-"*60)
    for model, metrics in results.items():
        print(f"{model:<20} {metrics['edge_score']:<15.4f} {metrics['mse']:<15.4f}")
    print("="*60)

if __name__ == '__main__':
    compare_all_models()

ModuleNotFoundError: No module named 'sklearn'